In [ ]:
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

: 

In [55]:
df_raw = pd.read_csv('../data/cps_00014.csv')
exposure_scores = pd.read_csv('../data/final_exposure.csv')

df_raw = df_raw[(df_raw['OCC'] != 0) & (df_raw['OCC'].notna())]

In [56]:
# CPS occupation
OCC_COL = 'OCC'
# CPS age
AGE_COL = 'AGE'
# Young/Entry Workers Column
Y_COL = '18_25'
# CPS weight  
WEIGHT_COL    = 'WTFINL'                  
# Scores occupation       
INDEX_OCC_COL = 'CPS Code'
# All score columns
INDEX_COLS = ['alpha', 'beta', 'gamma']

In [57]:
# Convert both OCC and CPS Code to string columns
df_raw[OCC_COL] = df_raw[OCC_COL].astype(float).astype(int).astype(str)
exposure_scores[INDEX_OCC_COL] = exposure_scores[INDEX_OCC_COL].astype(float).astype(int).astype(str)

In [58]:
# Merge exposure scores onto the dataset
df = df_raw.merge(exposure_scores,
                     left_on="OCC",
                     right_on=INDEX_OCC_COL,
                     how='left',
                     indicator=True)

# 32203 Rows do not merge
failed = df['_merge'] == 'left_only'
failed_to_merge = df[failed]

# 13 occupation codes did not merge
# They are residuals (All Other) or Military occupations (9840)
# Onet does not track them so they are being dropped
# https://www.onetcenter.org/dl_files/Taxonomy2019_Summary.pdf
# See citation 4 on page 10

assert failed_to_merge['OCC'].nunique() == 13, "Some rows did not merge or somehow not in ONET codes merged"

df = df.drop(columns=["CPS Code", "CPS Title", "_merge"])

df_scores = df[~failed]

In [59]:
# Construct panels at 4 different window lengths, check for cell size filters.
# Since the actual proportions per occupation are fairly low,
# Many low N and low n cells provide unreliable point estimates of the true proportion

# Create count of young workers
df_scores[Y_COL] = df_scores[AGE_COL].between(18, 25)

# Aggregate to quarter, half year and full year levels using treatment date as time = 0
treat_year = 2022
treat_month = 11

df_scores['MONTHS_FROM_TREAT'] = (df_scores["YEAR"] - treat_year) * 12 + (df_scores['MONTH'] - treat_month)
df_scores['QUARTER'] = df_scores['MONTHS_FROM_TREAT'] // 3
df_scores['HALF_YEAR'] = df_scores['MONTHS_FROM_TREAT'] // 6
df_scores['YEARS_FROM_TREAT'] = df_scores['MONTHS_FROM_TREAT'] // 12

# Drop trailing months when aggregating:

# Take number of months within every period specification and trim if the number of months is less than the specified window_length
# Tolerance -- we keep 2025H2 and 2025 but not 2025Q4 because of the October CPS
def drop_thin_periods(df, period_col, window_length, tolerance=1):
    counts = df.groupby(period_col)[['YEAR','MONTH']].apply(lambda g: g.drop_duplicates().shape[0])
    keep = counts[counts >= window_length - tolerance].index
    return df[df[period_col].isin(keep)]

df_q = drop_thin_periods(df_scores, 'QUARTER', 3, tolerance=0)
df_h = drop_thin_periods(df_scores, 'HALF_YEAR', 6, tolerance=1)
df_y = drop_thin_periods(df_scores, 'YEARS_FROM_TREAT', 12, tolerance=1)

In [60]:
# Function to get occupations that pass the filter in each period in each window size
def compute_cell_filters(df_w, window_col, n_min=30, np_min=5):
    #We only filter on pre-treatment data
    df_w = df_w[(df_w[window_col] < 0)]

    cells_w = df_w.groupby(['OCC', window_col]).agg(
        N=(AGE_COL, 'size'),
        np=(Y_COL, 'sum'))
    cells_w['nq'] = cells_w['N'] - cells_w['np']

    cell_counts_N = cells_w['N'].unstack(fill_value=0)
    cell_counts_np = cells_w['np'].unstack(fill_value=0)
    cell_counts_nq = cells_w['nq'].unstack(fill_value=0)

    valid = (cell_counts_N >= n_min) & (cell_counts_np >= np_min) & (cell_counts_nq >= np_min)
    passing_occ = valid.index[valid.all(axis=1)]
    pass_counts = valid.sum(axis=0)

    return {
        "passing_occ": passing_occ,            # List of OCCs that pass in ALL windows
        "pass_counts": pass_counts,            # Series: total passing OCCs per window
        "valid_matrix": valid,                 # Boolean DataFrame (OCC x Window)
        "cell_counts": {                       # Raw count matrices
            "N": cell_counts_N,
            "np": cell_counts_np,
            "nq": cell_counts_nq
        }
    }   

In [61]:
windows = {
    'monthly': (df_scores, 'MONTHS_FROM_TREAT'),
    'quarterly': (df_q, 'QUARTER'),
    'half-year': (df_h, 'HALF_YEAR'),
    'full-year': (df_y, 'YEARS_FROM_TREAT'),
}

filter_results = {}

for name, (df_w, window_col) in windows.items():
    filter_results[name] = compute_cell_filters(df_w, window_col)
    n_passing = len(filter_results[name]['passing_occ'])
    print(f"PASSING occupations ({name}): {n_passing}, Percent: {(n_passing/513)*100}%")
    

PASSING occupations (monthly): 126, Percent: 24.561403508771928%
PASSING occupations (quarterly): 252, Percent: 49.122807017543856%
PASSING occupations (half-year): 357, Percent: 69.5906432748538%
PASSING occupations (full-year): 465, Percent: 90.64327485380117%


In [ ]:
# Testing for selection bias
# Do cell-filter failing and passing occupations differ in their exposure scores

for index in INDEX_COLS:
    print(f'FOR INDEX {index}:')
    results_exposure = []
    for name, (df_w, window_col) in windows.items():
        df_w = df_w[(df_w[window_col] < 0)]
        passing_w = filter_results[name]['passing_occ']
        t_stat, p_val = stats.ttest_ind(
            df_w[df_w['OCC'].isin(passing_w)].groupby('OCC')[index].mean(),
            df_w[~df_w['OCC'].isin(passing_w)].groupby('OCC')[index].mean(),
            equal_var=False
        )
        results_exposure.append({'window': name, 't_stat': t_stat, 'p_val': p_val})
    results_exposure_df = pd.DataFrame(results_exposure)
    print(results_exposure_df)


# No statistically different results between filtered and unfiltered groups at any level across any index

FOR INDEX alpha:
      window    t_stat     p_val
0    monthly -0.301508  0.763310
1  quarterly  0.786377  0.432012
2  half-year  1.044432  0.297056
3  full-year  0.822624  0.414184
FOR INDEX beta:
      window    t_stat     p_val
0    monthly -0.372602  0.709810
1  quarterly  1.172261  0.241640
2  half-year  0.585569  0.558584
3  full-year -0.301922  0.763744
FOR INDEX gamma:
      window    t_stat     p_val
0    monthly -0.335120  0.737864
1  quarterly  1.136019  0.256483
2  half-year  0.250174  0.802620
3  full-year -0.852380  0.397381


In [63]:
results_share = []
for name, (df_w, window_col) in windows.items():
    df_w = df_w[(df_w[window_col] < 0)]
    passing_w = filter_results[name]['passing_occ']
    
    passing_vals = df_w[df_w['OCC'].isin(passing_w)].groupby('OCC')[Y_COL].mean()
    failing_vals = df_w[~df_w['OCC'].isin(passing_w)].groupby('OCC')[Y_COL].mean()
    t_stat, p_val = stats.ttest_ind(passing_vals, failing_vals, equal_var=False)
    diff = passing_vals.mean() - failing_vals.mean()
    
    # Baseline: average of occupation-level means across all occupations
    overall_occ_mean = df_w.groupby('OCC')[Y_COL].mean().mean()
    pct_diff = (diff / overall_occ_mean) * 100
    results_share.append({
        'window': name, 
        't_stat': t_stat, 
        'p_val': p_val, 
        'diff': diff, 
        'pct_diff': pct_diff
    })

print(pd.DataFrame(results_share))

# In every window filtered out cells have less young workers
# However based on CLT for bionmial proportions these estimates may not be accurate
# Will include high N groups in robustness check even if they are low np

      window     t_stat         p_val      diff   pct_diff
0    monthly   5.355090  2.485201e-07  0.051704  45.573860
1  quarterly   6.347027  4.931225e-10  0.048404  42.664808
2  half-year   8.749909  7.419858e-17  0.064429  56.790451
3  full-year  11.896127  5.320835e-20  0.093122  82.081377


In [64]:
for name, (df_w, window_col) in windows.items():
        df_w = df_w[(df_w[window_col] < 0)]
        emp_share = df_w.groupby('OCC')[WEIGHT_COL].sum()
        passing_w = filter_results[name]['passing_occ']
        print(f"Passing occ share of total employment of sample({name}: {(emp_share[passing_w].sum() / emp_share.sum())*100}%")

Passing occ share of total employment of sample(monthly: 73.0981533655647%
Passing occ share of total employment of sample(quarterly: 89.13520529874224%
Passing occ share of total employment of sample(half-year: 95.2493788933796%
Passing occ share of total employment of sample(full-year: 98.93637068128866%


In [ ]:
df_scores.to_csv('../data/checkpoints/df_scores.csv', index=False)
df_q.to_csv('../data/checkpoints/df_q.csv', index=False)
df_h.to_csv('../data/checkpoints/df_h.csv', index=False)

passing_occs_df = pd.concat([
    pd.DataFrame({'window': window_name, 'OCC': list(res['passing_occ'])})
    for window_name, res in filter_results.items()
], ignore_index=True)

passing_occs_df.to_json('../data/checkpoints/passing_occs.json', orient='records', indent=2)